# 03 KR2036 潮流、瓶頸與節點電價（kwak_full 單情境）

## 這本的定位

原本規劃是照 `validation/optimization_plots_nigeria_2020_2060.ipynb` 做
「基準情境 vs kwak_full」兩欄並排。**這個設計已經放棄**，原因在 02 第 7.3 節：
兩個情境同時有四項差異（年需求 667.3 對 707.0 TWh、VRE 容量 102.9 對 132.8 GW、
核能可用率 0.80 對 0.60、熱機組必發有無），並排會讓讀者把任何差異誤讀成單一成因。

改成 **kwak_full 單情境的輸電面深入分析**：潮流、瓶頸位置、節點電價。
保留原規劃中不受影響的兩張圖（負載分布地圖、輸電擴建地圖）。

**重要前提：本情境的輸電線路是自由擴建的。** 因此這裡看到的「瓶頸」是
**擴建之後仍然存在的殘餘瓶頸**，不是「現況電網的瓶頸」。模型已經把值得擴的線擴到
邊際效益等於邊際成本為止，殘餘壅塞代表的是「再擴下去不划算」而非「缺乏投資」。

**對偶變數的精度限制**：`config.yaml` 用 `solver: ipm` + `run_crossover: "off"`，
節點邊際價格是內點解的對偶值，非頂點解。用於比較大小趨勢沒問題，
但不宜直接引用到小數點後多位，也不適合拿去做結算金額的推算。

## 0. 環境設定

In [ ]:
import sys
import warnings
from pathlib import Path

_here = Path.cwd()
for cand in (_here, _here / "notebooks_kr", _here.parent):
    if (cand / "_kr_common.py").exists():
        sys.path.insert(0, str(cand))
        break

import _kr_common as K

K.setup_matplotlib()

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pypsa.plot import add_legend_circles, add_legend_lines

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

SCEN = "kwak_full"
info = K.SCENARIOS[SCEN]
n = K.load_network(SCEN)

# 沿用 scripts/plot_network.py 已修好的韓國尺度
BUS_SIZE_FACTOR = 2e6
LINEWIDTH_FACTOR = 1e4

lines = n.lines.copy()
lines["expand_MW"] = lines.s_nom_opt - lines.s_nom
lines["expand_%"] = (lines.s_nom_opt / lines.s_nom - 1) * 100
lines["名稱"] = lines.bus0 + " → " + lines.bus1

print("情境：", info["long_label"])
print(f"線路 {len(lines)} 條，s_nom {lines.s_nom.sum():,.0f} → s_nom_opt {lines.s_nom_opt.sum():,.0f} MW "
      f"(+{100 * (lines.s_nom_opt.sum() / lines.s_nom.sum() - 1):.1f}%)")
print(f"s_max_pu = {lines.s_max_pu.unique()}（論文同值；實際可用容量 = s_nom_opt × s_max_pu）")

## 1. 負載分布地圖

節點大小為該節點的年用電量（`loads_t.p_set` 以 `snapshot_weightings` 加權）。

In [ ]:
w = n.snapshot_weightings.generators
load_mwh = n.loads_t.p_set.mul(w, axis=0).sum()
load_by_bus = load_mwh.groupby(n.loads.bus).sum()

fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={"projection": ccrs.PlateCarree()})
n.plot(
    ax=ax,
    bus_sizes=load_by_bus / 6e7,
    bus_colors="#c44e52",
    bus_alpha=0.75,
    line_widths=n.lines.s_nom / LINEWIDTH_FACTOR,
    line_colors="#b0b0b0",
    link_widths=0,
    geomap=True,
    color_geomap={"ocean": "white", "land": "whitesmoke"},
    boundaries=K.KR_EXTENT,
)
for b, v in load_by_bus.items():
    ax.annotate(f"{v / 1e6:.0f}", xy=(n.buses.x[b], n.buses.y[b]), ha="center", va="center",
                fontsize=8, fontweight="bold", transform=ccrs.PlateCarree(), zorder=5,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
ax.set_title(f"KR2036 {info['label']}：年用電量分布（數字為 TWh）", fontsize=13, pad=10)

K.savefig("03_fig1_load_distribution")
plt.show()

share = (load_by_bus / load_by_bus.sum() * 100).sort_values(ascending=False)
pd.DataFrame({"年用電_TWh": load_by_bus / 1e6, "佔比%": share}).sort_values("年用電_TWh", ascending=False).round(2)

## 2. 輸電擴建地圖

線寬為最佳化後容量 `s_nom_opt`，顏色深淺表示擴建幅度。
節點大小為該節點的最佳化後發電容量，用來對照「發電集中處是否就是擴建集中處」。

In [ ]:
gen_by_bus = n.generators.groupby("bus").p_nom_opt.sum()

fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={"projection": ccrs.PlateCarree()})
n.plot(
    ax=ax,
    bus_sizes=gen_by_bus / BUS_SIZE_FACTOR,
    bus_colors="#4c72b0",
    bus_alpha=0.7,
    line_widths=lines.s_nom_opt / LINEWIDTH_FACTOR,
    line_colors=lines["expand_MW"],
    line_cmap="OrRd",
    link_widths=0,
    geomap=True,
    color_geomap={"ocean": "white", "land": "whitesmoke"},
    boundaries=K.KR_EXTENT,
)
sm = plt.cm.ScalarMappable(cmap="OrRd",
                           norm=plt.Normalize(0, float(lines["expand_MW"].max())))
cb = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
cb.set_label("線路擴建量 [MW]")
add_legend_lines(
    ax, [s / LINEWIDTH_FACTOR for s in (10e3, 40e3)], ["10 GW", "40 GW"],
    legend_kw=dict(loc="upper left", frameon=True, title="s_nom_opt"),
)
ax.set_title(f"KR2036 {info['label']}：輸電擴建分布", fontsize=13, pad=10)

K.savefig("03_fig2_transmission_expansion")
plt.show()

top = lines.sort_values("expand_MW", ascending=False)
top[["名稱", "s_nom", "s_nom_opt", "expand_MW", "expand_%"]].head(8).round(1)

## 3. 潮流與線路利用率

利用率定義為 `|p0| ÷ (s_nom_opt × s_max_pu)`，分母是**實際可用容量**
（`s_max_pu = 0.7`，與論文一致），因此 1.0 代表真的頂到限制。

In [ ]:
cap_eff = lines.s_nom_opt * lines.s_max_pu
util = n.lines_t.p0.abs().div(cap_eff, axis=1)

util_stats = pd.DataFrame({
    "名稱": lines["名稱"],
    "s_nom_opt_MW": lines.s_nom_opt,
    "擴建_MW": lines["expand_MW"],
    "平均利用率": util.mean(),
    "p90 利用率": util.quantile(0.90),
    "最大利用率": util.max(),
    "滿載時數比%": (util > 0.99).mean() * 100,
    "滿載時數": (util > 0.99).sum() * float(w.iloc[0]),
}).sort_values("滿載時數比%", ascending=False)

util_stats.round(3).to_csv(K.FIG_DIR / "03_table_line_utilisation.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "03_table_line_utilisation.csv")
print(f"全網平均利用率 {util.values.mean():.1%}；至少滿載過一次的線路 "
      f"{int((util.max() > 0.99).sum())}/{len(lines)} 條")
util_stats.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

ax = axes[0]
pct = np.linspace(0, 100, len(util))
congested = util_stats.index[util_stats["滿載時數比%"] > 0]
for col in util.columns:
    is_c = col in congested
    ax.plot(pct, np.sort(util[col].values)[::-1],
            color="#c44e52" if is_c else "#bbbbbb",
            linewidth=1.3 if is_c else 0.8, alpha=0.9 if is_c else 0.6,
            zorder=3 if is_c else 1)
ax.axhline(1.0, color="#333", linestyle="--", linewidth=1.0)
ax.text(1, 1.02, "可用容量上限（s_max_pu = 0.7）", fontsize=8, color="#333")
ax.set_xlabel("超越時間比例 [%]")
ax.set_ylabel("線路利用率")
ax.set_title("線路利用率持續曲線（紅色為曾滿載的線路）", fontsize=12)
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.3)

ax = axes[1]
s = util_stats
ax.scatter(s["平均利用率"], s["滿載時數比%"], s=s["s_nom_opt_MW"] / 120,
           c=s["擴建_MW"], cmap="OrRd", edgecolor="#555", linewidth=0.6, zorder=3)
for idx, row in s.iterrows():
    if row["滿載時數比%"] > 0.5 or row["平均利用率"] > 0.55:
        ax.annotate(row["名稱"], xy=(row["平均利用率"], row["滿載時數比%"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=7.5)
ax.set_xlabel("平均利用率")
ax.set_ylabel("滿載時數比 [%]")
ax.set_title("瓶頸定位（點大小＝容量，顏色＝擴建量）", fontsize=12)
ax.grid(alpha=0.3)

K.savefig("03_fig3_line_utilisation")
plt.show()

## 4. 瓶頸判讀：擴建之後還剩下什麼

這一節回答兩個問題：**殘餘瓶頸在哪裡**，以及**瓶頸發生時系統在做什麼**。

In [ ]:
# 瓶頸時段的系統狀態
any_congested = (util > 0.99).any(axis=1)
g = n.generators
vre = g.index[g.carrier.isin(["solar", "onwind", "offwind-ac", "offwind-dc"])]
avail = (n.generators_t.p_max_pu.reindex(columns=vre).fillna(g.loc[vre, "p_max_pu"])
         * g.loc[vre, "p_nom_opt"]).sum(axis=1)
actual = n.generators_t.p[vre].sum(axis=1)
curt = (avail - actual).clip(lower=0)
load_t = n.loads_t.p_set.sum(axis=1)
price_spread = n.buses_t.marginal_price.max(axis=1) - n.buses_t.marginal_price.min(axis=1)

state = pd.DataFrame({
    "時段數": [int(any_congested.sum()), int((~any_congested).sum())],
    "佔比%": [100 * any_congested.mean(), 100 * (~any_congested).mean()],
    "平均負載_GW": [load_t[any_congested].mean() / 1e3, load_t[~any_congested].mean() / 1e3],
    "平均 VRE 可發_GW": [avail[any_congested].mean() / 1e3, avail[~any_congested].mean() / 1e3],
    "平均棄電_GW": [curt[any_congested].mean() / 1e3, curt[~any_congested].mean() / 1e3],
    "平均跨節點價差": [price_spread[any_congested].mean(), price_spread[~any_congested].mean()],
}, index=["有線路滿載", "無線路滿載"])
state.round(2)

In [ ]:
# 棄電發生在哪個節點：節點棄電量 vs 該節點外送線路的滿載情形
curt_by_bus = (
    (n.generators_t.p_max_pu.reindex(columns=vre).fillna(g.loc[vre, "p_max_pu"])
     * g.loc[vre, "p_nom_opt"] - n.generators_t.p[vre])
    .clip(lower=0).mul(w, axis=0).sum()
    .groupby(g.loc[vre, "bus"]).sum()
)

# 每個節點所連線路的最大滿載時數比
bus_congestion = {}
for b in n.buses.index:
    conn = lines.index[(lines.bus0 == b) | (lines.bus1 == b)]
    bus_congestion[b] = float(util_stats.loc[conn, "滿載時數比%"].max()) if len(conn) else 0.0

bottleneck = pd.DataFrame({
    "VRE 棄電_GWh": curt_by_bus / 1e3,
    "外送線路最大滿載時數比%": pd.Series(bus_congestion),
    "VRE 容量_GW": g.loc[vre].groupby("bus").p_nom_opt.sum() / 1e3,
    "年均電價": n.buses_t.marginal_price.mean(),
}).fillna(0).sort_values("VRE 棄電_GWh", ascending=False)

bottleneck.round(2).to_csv(K.FIG_DIR / "03_table_bottleneck_by_bus.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "03_table_bottleneck_by_bus.csv")
bottleneck.round(2)

## 5. 節點電價

10 節點的年均電價與時間分布。價差是瓶頸最直接的經濟訊號：
在無損耗的線性潮流下，**兩節點價格不同就代表其間的輸電受限**。

In [ ]:
price = n.buses_t.marginal_price
price_mean = price.mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 8),
                         subplot_kw={"projection": ccrs.PlateCarree()})

ax = axes[0]
ax.set_extent(K.KR_EXTENT, crs=ccrs.PlateCarree())
regions = K.load_regions(10, "onshore")
regions["price"] = price_mean.reindex(regions.index)
regions.plot(ax=ax, column="price", cmap="RdYlBu_r", linewidth=0.5, edgecolor="k",
             legend=True, legend_kwds={"label": "年均節點電價 [EUR/MWh]", "shrink": 0.6},
             transform=ccrs.PlateCarree())
for b, v in price_mean.items():
    ax.annotate(f"{v:.1f}", xy=(n.buses.x[b], n.buses.y[b]), ha="center", va="center",
                fontsize=8, fontweight="bold", transform=ccrs.PlateCarree(), zorder=5,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
ax.set_title("年均節點電價", fontsize=12, pad=8)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
gl.top_labels = False
gl.right_labels = False

# 右圖畫「價差」而不是十條幾乎重疊的節點價格曲線：
# 93.8% 的時段全網單一價格，十條線會疊成一條，看起來像畫錯而不像發現。
ax = axes[1]
ax.remove()
ax = fig.add_subplot(1, 2, 2)
pct = np.linspace(0, 100, len(price_spread))
ax.plot(pct, np.sort(price_spread.values)[::-1], color="#c44e52", linewidth=1.8)
ax.fill_between(pct, 0, np.sort(price_spread.values)[::-1], color="#c44e52", alpha=0.18)

share_congested = 100 * float((price_spread > 1).mean())
ax.axvline(share_congested, color="#333", linestyle="--", linewidth=1.0)
ax.annotate(
    f"僅 {share_congested:.1f}% 的時段有價差；其餘時段全網單一價格",
    xy=(share_congested, price_spread.max() * 0.55),
    xytext=(share_congested + 12, price_spread.max() * 0.72),
    fontsize=9, arrowprops=dict(arrowstyle="->", color="#333", linewidth=0.9),
)
ax.set_xlabel("超越時間比例 [%]")
ax.set_ylabel("跨節點最大價差 [EUR/MWh]")
ax.set_title("跨節點價差持續曲線", fontsize=12)
ax.grid(alpha=0.3)

fig.suptitle(f"KR2036 {info['label']}：節點電價", fontsize=14)
K.savefig("03_fig4_nodal_price")
plt.show()

print(f"年均電價：最低 {price_mean.min():.2f}（{price_mean.idxmin()}）"
      f"　最高 {price_mean.max():.2f}（{price_mean.idxmax()}）"
      f"　價差 {price_mean.max() - price_mean.min():.2f} EUR/MWh")
print(f"同一時刻跨節點價差 > 1 EUR/MWh 的時段：{100 * (price_spread > 1).mean():.1f}%")
print(f"跨節點價差的 p50 / p90 / 最大：{price_spread.quantile(.5):.2f} / "
      f"{price_spread.quantile(.9):.2f} / {price_spread.max():.2f} EUR/MWh")

In [ ]:
# 價差與壅塞是否同時發生
comp = pd.DataFrame({
    "跨節點價差": price_spread,
    "滿載線路數": (util > 0.99).sum(axis=1),
    "棄電_GW": curt / 1e3,
})
print("相關係數：")
print(comp.corr().round(3).to_string())
print()
print("依滿載線路數分組的平均跨節點價差：")
print(comp.groupby("滿載線路數")[["跨節點價差", "棄電_GW"]].agg(["mean", "count"]).round(3).to_string())

In [ ]:
grp = comp.groupby("滿載線路數")["跨節點價差"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(grp.index.astype(int), grp["mean"], color="#c44e52", edgecolor="white", width=0.7)
for xi, (mean_v, cnt) in zip(grp.index, grp[["mean", "count"]].values):
    ax.text(xi, mean_v + 3, f"{mean_v:.0f}", ha="center", fontsize=9)
    ax.text(xi, -6, f"n={int(cnt)}", ha="center", fontsize=7.5, color="#666")
ax.set_xlabel("同一時段內滿載的線路數")
ax.set_ylabel("平均跨節點價差 [EUR/MWh]")
ax.set_title(f"價差由壅塞驅動（相關係數 {comp['跨節點價差'].corr(comp['滿載線路數']):.3f}）", fontsize=13, pad=10)
ax.set_ylim(-12, float(grp["mean"].max()) * 1.2)
ax.grid(axis="y", alpha=0.3)

K.savefig("03_fig5_spread_vs_congestion")
plt.show()

## 6. 結論

（數字以上方 cell 的實際輸出為準。）

1. **擴建是為了把發電外送，不是把電送進負載中心。**
   `KR0 8` 有 32.5 GW 的 VRE 容量，年用電量卻只佔全國 1.3%（9.0 TWh）；
   它的外送線路 `KR0 8 → KR0 9` 擴建了 **+36,532 MW（+1,769%）**，是全網最大的一筆。
   擴建量前三名全部連到 `KR0 9`。相對地，最大負載節點 `KR0 6`（180.8 TWh，佔 25.6%）
   所連線路的擴建幅度只有 +39%。**這是發電端外送瓶頸，不是負載端受電瓶頸。**

2. **殘餘壅塞集中在高 VRE 時段。** 17 條線有 11 條至少滿載過一次，
   但全網平均利用率只有 45.7%，滿載時數比都在 3% 以下。
   有線路滿載的 181 個時段（6.2%）平均 VRE 可發 **51.3 GW**，
   其餘時段只有 **26.4 GW**——差距近兩倍，壅塞幾乎只發生在風光大發的時候。
   注意**線路是自由擴建的**，所以最適解本來就不該留下大量壅塞；
   這個結果**不能解讀成「韓國電網很寬裕」**，只能說「擴建 45.3% 之後再擴不划算」。

3. **空間價差是「全有全無」的，不要用年均價差下結論。**
   93.8% 的時段跨節點價差為 **0.001 EUR/MWh**（等於沒有價差）——
   這正是理論預期：無壅塞時全網單一價格，模型行為正確。
   但在壅塞時段，平均價差高達 **63.4 EUR/MWh**，最大 124.8。
   價差與滿載線路數的相關係數 **0.820**，且隨滿載線路數單調上升
   （1 條 → 20.4、3 條 → 96.1、8 條 → 118.8 EUR/MWh）。

   **年均價差 3.41 EUR/MWh 是把 6.2% 時段的巨大價差稀釋到全年的結果**，
   用它說「價差很小」會嚴重誤導。正確說法是：**價差在時間上高度集中。**

4. **棄電與輸電壅塞無關——這一點違反直覺，但數據很明確。**
   棄電與跨節點價差的相關係數是 **−0.003**（等於零）；
   無壅塞時段平均棄電 1.11 GW，有壅塞時段 1.31 GW，幾乎沒有差別。
   代表全年 9.79 TWh 的 VRE 棄電**來自時間維度的過剩**（必發下限 + 供過於求，
   見 02 第 7 節），**不是空間維度的「送不出去」**。
   因此**再擴輸電線並不會顯著減少棄電**——這對「要不要繼續投資電網」的政策討論
   是關鍵區分。要減少這部分棄電，要處理的是必發限制與儲能，不是線路。

5. **10 節點的先天限制。** 只有 10 個節點、17 條線，濟州併入本土叢集，
   因此「瓶頸位置」只能定位到大區級別，無法對應到實際變電所或走廊。
   要談具體線路的強化順序需要 30 節點以上的結果
   （`solved/30n_3H_CCL.nc` 存在，但那是基準情境設定，與本情境不可直接比較）。